#  Optimizing a Model with LLM Compressor

Intro: [moving from the slide]

Old intro:
In the previous lesson, you learned about quantization: what it is, why it matters, and how they reduce the cost of running AI models. In this lesson, you're going to actually do it. You'll take a full-precision model, compress it with a tool called llm-compressor, compare the sizes, and then measure whether the model still works well. Time to compress some weights.

In [ ]:
         1         2         3         4         5         6         7  
#23456789012345678901234567890123456789012345678901234567890123456789012

## Setup

First, we'll import our standard libraries, like torch, the Hugging Face transformers for loading models, and tokenizers.

We're working with Qwen3 .6B as our base model. It's small enough to work with in this environment, but it's a real, capable language model, and you've got folders for both the original and quantized model weights.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MODEL_DIR = "Qwen3-0.6B"
OUTPUT_DIR = "Qwen3-0.6B-W4A16"

print(f"Base model:      {MODEL_DIR}")
print(f"Quantized model: {OUTPUT_DIR}")

Base model:      Qwen3-0.6B
Quantized model: Qwen3-0.6B-W4A16


## Define the Recipe


First you need to specify a recipe which tells LLM Compressor how to quantize. This includes the algorithm to use for quantization along with the scheme, meaning how many bits do we want to compress the model to. 

We're using the GPTQ algorithm today, using calibration data to find the optimal quantized values for each weight. Then, we're going for 4-bit weights with 16-bit activations, which can lead to a rougly 50% total reduction. We're targeting linear layers, where the vast majority of parameters live, and ignoring the lm_head, the output layer that maps tokens to vocabulary so we keep this compressed model precise.

A recipe could be list of quantization algorithms, but for here we're using one algorithm.

In [2]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False sequential_targets=None block_size=128 dampening_frac=0.01 actorder=static offload_hessians=False


## Quantize the Model

Now to produce the the quantized model, we need to impport `oneshot`. It takes the model, some calibration data, and a recipe together to do compression in a single pass. 

The calibration dataset, which we use during quantization to make the model significantly more accurate than say, a round to nearest for the models weights. GPTQ and other algorithms runs the calibration data through the model, to measure how sensitive the output is to each weight, then finds quantized values that minimize the overall rounding error. Weights that strongly affect predictions are preserved carefully; less important ones absorb more of the rounding error. The result is a 4-bit model that behaves much closer to the original than naive rounding would give you.

The `dataset` parameter specifies what text to use for calibration. Here we use [WikiText-2](https://huggingface.co/datasets/mindchain/wikitext2), a standard benchmark of Wikipedia articles, the same dataset you'll use later for perplexity evaluation. 

`max_seq_length` and `num_calibration_samples` control that calibration pass:
- number of calibration samples: is how many sequences are run through the model. More samples give a better picture of weight importance, but past a few hundred the accuracy gains become tiny while runtime keeps growing. 256 is a solid default.
- Max sequence length is the max token length per sample. Longer sequences let the quantizer see how weights behave across realistic context lengths; samples beyond this get truncated.

These parameters help to improve accuracy as we go from default full precision (BFloat16) to Int 4 precision of weights.

Since quantization can take several minutes and benefits from a GPU, we've already run it ahead of time and provided the quantized model in the `Qwen3-0.6B-W4A16` folder (`OUTPUT_DIR`). The `if not os.path.isdir(OUTPUT_DIR)` check above ensures we skip re-running quantization when the folder already exists, so you can move straight to evaluation. 

> We'll show you real quick what this oneshot looks like (speed up)


In [3]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=4096,
        num_calibration_samples=256,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

## Compare Model Sizes

Awesome, let's see what quantization actually saved us. This cell walks through both model directories, both the original and compressed that you already have in your environment, and prints a comparison. You're saving already about 50% in just size of the model. 

You might expect a 75% reduction since we went from 16-bit to 4-bit weights (4x smaller), but the actual reduction is 42%. The reason: only the **linear layer weights** are quantized to Int4. The rest of the model (including the LM head and normalization layers) stays at higher precision. 

So the 4x compression applies to the bulk of the parameters (the linear layers, which dominate the model), but the unquantized pieces  pull the overall reduction down to ~42%. This ratio improves with larger models, where linear weights make up an even bigger share of total size — a 70B model quantized the same way gets much closer to the theoretical 4x.

In [4]:
def folder_size(path):
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(nbytes):
    if nbytes < 1024**2:
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3:
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig = folder_size(MODEL_DIR)
size_q = folder_size(OUTPUT_DIR)
reduction = (1 - size_q / size_orig) * 100 if size_orig > 0 else 0

print("Model Size Comparison")
print("=" * 45)
print(f"Original (BF16):    {format_size(size_orig)}")
print(f"Quantized (W4A16):  {format_size(size_q)}")
print(f"Reduction:          {reduction:.0f}%")

Model Size Comparison
Original (BF16):    1.41 GB
Quantized (W4A16):  835.3 MB
Reduction:          42%


## Test Both Models

Smaller model sizes are only useful if the model still works, so let's do a comparison of both using the same prompt. First, the base model, which we'll load in it's original weights on CPU, and give it a prompt "machine learning is a branch of..." and let it generate 60 tokens with standard sampling settings. This is our baseline. Once we're done, we'll get remove the original model from memory and load in the quantized one.

In [5]:
prompt = "Machine learning is a branch of"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = base_model.generate(
    **inputs, 
    max_new_tokens=60, 
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Base Model ({MODEL_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, 
                                    skip_special_tokens=True)}")

#del base_model; gc.collect()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Base Model (Qwen3-0.6B)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has gained significant attention in recent years, particularly in the context of the rise of big data and the need for efficient, scalable solutions to complex problems. As the field continues to evolve, the integration of machine learning into various industries is becoming increasingly widespread. However, despite its growing popularity,


Now we're using the quantized model with the exact same prompt and generation settings. The only difference is that the model has 4-bit weights instead of 16-bits. 


1. lets ru the cell
2. itll take some time
3. heres the output
4. lets compare between



Compare the two outputs, the quantized model should produce output similar to the baseline, maybe not word for word, but similar to the baseline. Huge compression, but minimal quality loss. that's the goal!



In [6]:
import logging
logging.getLogger("llmcompressor").setLevel(logging.WARNING)

quant_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = quant_model.generate(
    **inputs, 
    max_new_tokens=60, 
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Quantized Model ({OUTPUT_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, 
                                    skip_special_tokens=True)}")

Compressing model: 196it [00:00, 3156.34it/s]


Quantized Model (Qwen3-0.6B-W4A16)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that focuses on the development of algorithms and models that can learn from data to make predictions or decisions. It has become increasingly popular in various fields, including healthcare, finance, and marketing. However, the field is still in its early stages, and there are many challenges that need to be addressed


## Perplexity Comparison

Side-by-side text gives you intuition, but we need a number. That's where perplexity comes in: a standard metric for language models that measures how well the model predicts text. Lower is better, and if quantization has degraded the model, perplexity will be noticeably higher.

Here we define a `calculate_perplexity` function that takes a chunk of text from the WikiText-2 test set, slides a window across it, and at each position computes the cross-entropy loss between the model's predicted next-token distribution and the actual next token, essentially, how surprised the model is by the token that actually came next. Exponentiating the average loss gives us perplexity.

Each window moves forward by `stride` tokens and overlaps with the previous one. The sliding window with stride lets us evaluate long text without feeding it all in at once, while still giving each token a reasonable amount of left context. 

We load the test split: same dataset family as calibration, but a held-out portion, so there's no data leakage.

In [7]:
from datasets import load_dataset

def calculate_perplexity(
        model, tokenizer, dataset, max_tokens=5000, stride=512):
    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
print(f"Loaded {len(test_data)} test samples")

Loaded 4358 test samples


Now, we compute perplexity for the quantized model first, as its already in memory. It'll take a moment, as we're running the full test data through the model. Once it's done, we'll clean the model from memory to test the base model

In [8]:
quant_ppl = calculate_perplexity(quant_model, tokenizer, test_data)
print(f"Quantized perplexity: {quant_ppl:.2f}")

Quantized perplexity: 35.48


With the base model, we compute its perplexity on the same test data. It gives us a reference point. 

In [10]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)
base_ppl = calculate_perplexity(base_model, tokenizer, test_data)
print(f"Base perplexity: {base_ppl:.2f}")

Base perplexity: 32.79


Finally, we have the comparison, base perplexity and quantized perplexity side by side. Now, a small increase is completely expected. We went from 16-bit weights down to 4-bit. The question is whether the increase is acceptable for your use case. For most production deployments, a few percent increase in perplexity is well worth the reduction in model size and the infrastructure savings that come with it.

In [11]:
print("Perplexity Comparison")
print("=" * 40)
print(f"Base (BF16):       {base_ppl:.2f}")
print(f"Quantized (W4A16): {quant_ppl:.2f}")
print(f"Difference:        {quant_ppl - base_ppl:+.2f} ({(
    quant_ppl/base_ppl - 1)*100:+.1f}%)")

Perplexity Comparison
Base (BF16):       32.79
Quantized (W4A16): 35.48
Difference:        +2.70 (+8.2%)


Optional: Let's recap. You learned how the oneshot API applies post-training quantization with a GPTQ recipe. You compared model sizes and saw the concrete reduction from W4A16. You tested both models on the same prompt to verify the output still makes sense. And you measured perplexity to put a number on the accuracy tradeoff.
In the next lesson, you'll take a model and serve it with vLLM — setting up the inference server and interacting with it through the OpenAI-compatible API. And in lesson five, you'll benchmark it under load and compare performance between the base and quantized versions. See you there.